In [1]:
import os
import pandas as pd
import numpy as np
import yaml
import pickle

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier

from utils.training_utils import find_specific_variables
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(os.path.join('..', 'data', 'train_test', 'train_encoded.csv'))

print(df.shape)
df.head()

(32940, 14)


,contact,default,education,job,marital,month,poutcome,quarter,age,campaign,contacts_tendency,pdays,previous,y
0,0.0,0.0,6.0,0.0,2.0,9.0,1.0,2.0,31.0,3.0,0.0,999.0,0.0,0
1,1.0,0.0,3.0,3.0,1.0,6.0,1.0,1.0,39.0,2.0,0.0,999.0,0.0,0
2,0.0,0.0,5.0,2.0,1.0,3.0,1.0,2.0,34.0,4.0,0.0,999.0,0.0,0
3,1.0,1.0,2.0,9.0,1.0,6.0,1.0,1.0,36.0,9.0,0.0,999.0,0.0,0
4,0.0,1.0,7.0,8.0,2.0,1.0,1.0,2.0,25.0,1.0,0.0,999.0,0.0,0


In [3]:
features = yaml.safe_load(open(os.path.join('..', 'src', 'config', 'feature_config.yaml'), 'r'))
feature_target = find_specific_variables(features, 'target', specific_value=True)

In [4]:
feature_target

['y']

In [5]:
seletor = pickle.load(
    open(os.path.join('..', 'models', 'encoders', 'seletor_2.pkl'), 'rb')
)

seletor.features

['age',
 'campaign',
 'contact',
 'contacts_tendency',
 'default',
 'education',
 'job',
 'marital',
 'month',
 'pdays',
 'poutcome',
 'previous',
 'quarter']

In [6]:
scale_pos_weight = df[df[feature_target]==0].shape[0] / df[df[feature_target]==1].shape[0]

models = {
    'DT': DecisionTreeClassifier(class_weight='balanced'),
    'RF': RandomForestClassifier(class_weight='balanced'),
    'GBT': GradientBoostingClassifier(),
    'ADA': AdaBoostClassifier(),
    'XGBoost': XGBClassifier(scale_pos_weight=scale_pos_weight),
    'LGBM': LGBMClassifier(class_weight='balanced')
}

In [7]:
sampling_strategies = {
    'Original': None,
    'Undersampling': RandomUnderSampler(random_state=96),
    'Oversampling': RandomOverSampler(random_state=96),
    'SMOTE': SMOTE(random_state=96)
}

results_sampling = {}

for sampling_name, sampler in sampling_strategies.items():
    print(f'\nSampling Strategy: {sampling_name}')
    results_sampling[sampling_name] = {}
    
    for model_name, model in models.items():
        skf = StratifiedKFold(n_splits=5, random_state=96, shuffle=True)
        
        if sampler is not None:
            pipeline = Pipeline([
                ('sampler', sampler),
                ('classifier', model)
            ])
        else:
            pipeline = model

        scores = cross_val_score(
            pipeline,
            df[seletor.features],
            df[feature_target],
            cv=skf,
            scoring='roc_auc'
        )

        results_sampling[sampling_name][model_name] = scores

        print(f'{model_name}: {np.mean(scores):.4f} +/- {np.std(scores):.4f}')



Sampling Strategy: Original


NameError: name 'StratifiedKFold' is not defined